## 1 setup

In [1]:
%pip install -q "datasets==3.6.0" pandas pyarrow numpy scipy scikit-learn tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: C:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import ast #per convertire liste e diz python in oggetti veri
import json 
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display #per mostrare meglio tabelle e DataFrame
from scipy.sparse import csr_matrix #per matrici sparse
from sklearn.decomposition import TruncatedSVD #per fare decomposizione della matrice utente-prodotto
from sklearn.neighbors import NearestNeighbors #per trovare elementi simili
from tqdm.auto import tqdm

pd.set_option("display.max_columns",80)
pd.set_option("display.max_colwidth",120)

C:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CATEGORY="Electronics"
RANDOM_STATE=42
K_VALUES=(5,10,20)
PRIMARY_METRIC="ndcg" 
#dobbiamo usare ndcg come metrica, perchè misura quanto bene il modello ordina gli elementi
#nelle prime k posizioni
#sommatoria per le prime k pos di (rilevanza elemento in posizione i/log2(i+1))
#se prodotto fuori dalla topk è uguale a 0
PRIMARY_K=10
#scegli che modelli seguire
RUN_POPULARITY=True
RUN_ITEMKNN=True
RUN_SVD=True

USE_RATING_VALUES=False #stimiamo se utente interagisce col prodotto, non il rating che deve dare
MAX_EVAL_ROWS=None #quante righe usare al max nella valutazione, con none le usi tutte
SAVE_RANKS=True #salva i rank prodotti dai modelli
SAVE_MODEL_ARTIFACTS=True #salva artefatti,mapping..

ITEMKNN_NEIGHBORS_LIST=[5,10,50,100] #guarda 50 prodotti vicini/simili
ITEMKNN_N_JOBS=-1 #usa tutte le risorse disponibili

SVD_FACTORS=[2,4,6,8,10,12,16,24,32] #fattori latenti da provare per truncatedSVD per approssimare la matrice utenti-prodotti
SVD_N_ITER=7 #num iterazioni di svd

DEMO_USERS_N=3 #3 utenti demo
DEMO_TOP_N=10 #mostra 10 raccomandazioni per utente

## 2 percorsi e caricamento dati

In [4]:
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
RECOMMENDER_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_recommender"
NOTEBOOK04_DIR=RECOMMENDER_DIR/"notebook04_models"
NOTEBOOK04_DIR.mkdir(parents=True,exist_ok=True)

TRAIN_PATH=RECOMMENDER_DIR/"train.parquet"
VALIDATION_EVAL_PATH=RECOMMENDER_DIR/"validation_eval.parquet"
TEST_EVAL_PATH=RECOMMENDER_DIR/"test_eval.parquet"
VALIDATION_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_validation.parquet"
TEST_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_test.parquet"
ITEMS_PATH=RECOMMENDER_DIR/"items.parquet"
USER_MAPPING_PATH=RECOMMENDER_DIR/"user_mapping.parquet"
ITEM_MAPPING_PATH=RECOMMENDER_DIR/"item_mapping.parquet"
SPLIT_SUMMARY_PATH=RECOMMENDER_DIR/"split_summary.json"

print("Project root:",PROJECT_ROOT)
print("Recommender dir:",RECOMMENDER_DIR)
print("Notebook04 dir:",NOTEBOOK04_DIR)

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender
Notebook04 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\notebook04_models


In [5]:
train_df=pd.read_parquet(TRAIN_PATH)
validation_eval_df=pd.read_parquet(VALIDATION_EVAL_PATH)
test_eval_df=pd.read_parquet(TEST_EVAL_PATH)
validation_samples_df=pd.read_parquet(VALIDATION_NEGATIVE_PATH)
test_samples_df=pd.read_parquet(TEST_NEGATIVE_PATH)
items_df=pd.read_parquet(ITEMS_PATH)
user_mapping_df=pd.read_parquet(USER_MAPPING_PATH)
item_mapping_df=pd.read_parquet(ITEM_MAPPING_PATH)
split_summary=json.loads(SPLIT_SUMMARY_PATH.read_text(encoding="utf-8"))

loaded_shapes=pd.DataFrame([
    {"name":"train","rows":len(train_df),"cols":train_df.shape[1]},
    {"name":"validation_eval","rows":len(validation_eval_df),"cols":validation_eval_df.shape[1]},
    {"name":"test_eval","rows":len(test_eval_df),"cols":test_eval_df.shape[1]},
    {"name":"validation_samples","rows":len(validation_samples_df),"cols":validation_samples_df.shape[1]},
    {"name":"test_samples","rows":len(test_samples_df),"cols":test_samples_df.shape[1]},
    {"name":"items","rows":len(items_df),"cols":items_df.shape[1]},
    {"name":"user_mapping","rows":len(user_mapping_df),"cols":user_mapping_df.shape[1]},
    {"name":"item_mapping","rows":len(item_mapping_df),"cols":item_mapping_df.shape[1]},
])

display(loaded_shapes)
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]).head(30))

,name,rows,cols
0,train,287752,14
1,validation_eval,131376,14
2,test_eval,131352,14
3,validation_samples,131376,7
4,test_samples,131352,7
5,items,12306,13
6,user_mapping,131398,2
7,item_mapping,12306,2


,value
category,Electronics
source_config,user3_item8
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,100
random_state,42
temporal_column,datetime
original_candidate_interactions,844060
positive_interactions_before_refilter,671903


In [6]:
def parse_negative_items(value): #prende valore della colonna negative_item_idxs e prova a convertirlo in lista
    if value is None:
        return []
    if isinstance(value,np.ndarray): #se array lo converte in lista python
        value=value.tolist()
    if isinstance(value,(list,tuple,set)): #se già lista, converte ogni elemento in int
        return [int(item) for item in value]
    if isinstance(value,str): #se valore è una stringa, toglie spazi inutili e prova a interpretarla come lista python
        value=value.strip()
        if not value:
            return []
        parsed=ast.literal_eval(value)
        return [int(item) for item in parsed]
    try:
        if pd.isna(value):
            return []
    except (TypeError,ValueError):
        pass
    return [int(value)]

validation_samples_df=validation_samples_df.copy()
test_samples_df=test_samples_df.copy()

validation_samples_df["negative_item_idxs"]=validation_samples_df["negative_item_idxs"].apply(parse_negative_items)
test_samples_df["negative_item_idxs"]=test_samples_df["negative_item_idxs"].apply(parse_negative_items)

if MAX_EVAL_ROWS is not None: #se non nullo, limita righe usate per valutazione
    validation_samples_df=validation_samples_df.head(int(MAX_EVAL_ROWS)).copy()
    test_samples_df=test_samples_df.head(int(MAX_EVAL_ROWS)).copy()

print("Validation samples:",len(validation_samples_df))
print("Test samples:",len(test_samples_df))
display(validation_samples_df.head())

Validation samples: 131376
Test samples: 131352


,split,user_idx,user_id,positive_item_idx,positive_parent_asin,negative_item_idxs,num_negatives
0,validation,0,AE22236AFRRSMQIKGG7TPTB75QEA,4617,B003ZDO2LM,"[259, 3414, 5268, 10203, 6251, 6543, 3981, 11457, 3205, 9702, 5687, 6513, 6868, 1934, 11272, 5301, 5897, 8066, 5849,...",100
1,validation,1,AE222H3FGXWLHRFUMGMS2RR57NDQ,6050,B0054JJ0QW,"[7710, 7535, 9359, 6955, 9631, 3850, 11767, 2880, 10843, 9619, 10135, 10058, 1190, 8283, 11116, 4856, 624, 1282, 100...",100
2,validation,2,AE223NW3QAU2KTAY5MAEU6VPBAUA,5807,B004VRQNS6,"[7396, 842, 654, 6114, 7341, 9000, 11775, 6275, 8528, 1200, 3251, 189, 2318, 637, 8829, 9776, 1865, 5393, 10232, 103...",100
3,validation,3,AE224HM2QAW5TTSDL2QRDERA6KMA,343,B0000DBJ8A,"[7445, 9637, 11790, 8567, 7608, 6313, 8444, 2044, 3338, 4990, 87, 6805, 7189, 1710, 3695, 5050, 5386, 8303, 11151, 7...",100
4,validation,4,AE2252DKW4XJIZP5QPFMQVJBVRTA,2444,B001L5SN7U,"[10098, 5608, 9372, 12247, 5178, 4924, 4000, 351, 5878, 3300, 6104, 1334, 14, 1952, 10369, 4984, 1346, 3484, 19, 182...",100


## 3 matrice utente prodotto

In [7]:
#calcoliamo numero totale di utenti e prodotti
n_users=int(max(user_mapping_df["user_idx"].max(),train_df["user_idx"].max())+1) 
n_items=int(max(item_mapping_df["item_idx"].max(),train_df["item_idx"].max())+1)
#estraiamo dal training set due array numpy
train_user_idx=train_df["user_idx"].astype(np.int64).to_numpy()
train_item_idx=train_df["item_idx"].astype(np.int64).to_numpy()

if USE_RATING_VALUES and "rating" in train_df.columns: #se true, allora prendi rating dal dataset
    train_values=train_df["rating"].astype(np.float32).to_numpy()
else:
    train_values=np.ones(len(train_df),dtype=np.float32) #mette 1 se esiste l'interazione, non ci interessa il rating

#costruisci matrice sparsa, righe=utenti, colonne=prodotti, valori=interazioni
train_matrix=csr_matrix((train_values,(train_user_idx,train_item_idx)),shape=(n_users,n_items),dtype=np.float32)
train_matrix.sum_duplicates() #se ci sono duplicati, li somma
train_matrix.eliminate_zeros() #pulisce zero salvati

#Verifica delle interazioni positive binarie
train_nonzero_values=np.unique(train_matrix.data)
if not USE_RATING_VALUES:
    if train_matrix.nnz!=len(train_df):
        raise ValueError("Duplicati utente-item nel training: verificare prima di valutare ItemKNN")
    if not np.array_equal(train_nonzero_values,np.array([1.0],dtype=np.float32)):
        raise ValueError("La matrice del training deve contenere solo valori non nulli uguali a 1")

density=train_matrix.nnz/(n_users*n_items) #calcola densità matrice, nnz=numero celle non nulle

matrix_stats={
    "n_users":int(n_users),
    "n_items":int(n_items),
    "train_interactions":int(train_matrix.nnz),
    "train_rows":int(len(train_df)),
    "nonzero_min":float(train_matrix.data.min()),
    "nonzero_max":float(train_matrix.data.max()),
    "nonzero_values":train_nonzero_values.tolist(),
    "density_pct":float(density*100),
}

display(pd.DataFrame.from_dict(matrix_stats,orient="index",columns=["value"]))

,value
n_users,131398
n_items,12306
train_interactions,287752
train_rows,287752
nonzero_min,1.0
nonzero_max,1.0
nonzero_values,[1.0]
density_pct,0.017796


In [8]:
#crea dizionario, per ogni utente crea l'insieme dei prodotti visti nel train
train_seen_by_user=(
    train_df.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#unisce in un unico dataframe le interazioni di train,val,test e crea una tabella con tutte le int note dell'utente
all_interactions_for_seen=pd.concat([train_df,validation_eval_df,test_eval_df],ignore_index=True)
#crea altro dizionario, per ogni utente, insieme di tutti i prodotti visti in train+val+test
all_seen_by_user=(
    all_interactions_for_seen.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#crea array con tutti prodotti presenti nel training set
train_item_candidates=np.array(sorted(train_df["item_idx"].astype(int).unique()),dtype=np.int64)

print("Utenti con storico train:",len(train_seen_by_user))
print("Item candidati dal train:",len(train_item_candidates))

Utenti con storico train: 131398
Item candidati dal train: 12299


## 4 funzioni di valutazione

In [9]:
#calcola in che posizione viene messo il prodotto positivo rispetto ai negativi campionati
def rank_positive(user_idx,positive_item_idx,negative_item_idxs,score_fn):
    #costruisce lista dei prodotti candidati da ordinare
    candidate_items=[int(positive_item_idx)]+[int(item) for item in negative_item_idxs]
    #modello assegna punteggio a ogni prodotto candidato
    scores=score_fn(int(user_idx),candidate_items)
    scores=np.asarray(scores,dtype=np.float64) #converte punteggi in array numpy
    scores=np.nan_to_num(scores,nan=-np.inf,posinf=np.inf,neginf=-np.inf)
    positive_score=float(scores[0]) #punteggio del nostro prodotto positivo
    rank=1
    for item_idx,score in zip(candidate_items[1:],scores[1:]): #scorre solo i prodotti negativi , con [1:] salta il primo
        if score>positive_score: #se prod neg ha punteggio piu alto, peggiora di uno il rank del positivo
            rank+=1
        elif score==positive_score and int(item_idx)<int(positive_item_idx): #vince item con id piu piccolo
            rank+=1
    return int(rank),positive_score #rank=posizione del prodotto tra i negativi+positivo, positive_score=punteggio assegnato

In [10]:
#prende una lista dei rank ottenuti, e i valori di k da valutare
def calculate_topk_metrics(ranks,k_values):
    ranks_array=np.asarray(ranks,dtype=np.float64) #converte lista in array numpy
    rows=[]
    for k in k_values: #scorre tutti i k
        hits=ranks_array<=int(k) #calcola per ogni esempio se il prodotto positivo è entrato in topk
        hitrate=float(np.mean(hits)) if len(ranks_array) else np.nan #calcola % casi in cui il positivo è in topk
        ndcg_values=np.where(hits,1.0/np.log2(ranks_array+1.0),0.0) #calcola valore ndcg@k per ogni esempio
        #where- se hits true calcola la funzione, se è false mette 0
        mrr_values=np.where(hits,1.0/ranks_array,0.0) #calcola mrr#k cioè mean reciprocal rank
        #misura quanto in alto il modello posiziona il prodotto positivo reale, 1/rank
        rows.append({
            "K":int(k),
            "hitrate":hitrate,
            "recall":hitrate,
            "ndcg":float(np.mean(ndcg_values)) if len(ranks_array) else np.nan,
            "mrr":float(np.mean(mrr_values)) if len(ranks_array) else np.nan,
            "n_examples":int(len(ranks_array)),
        })
    return pd.DataFrame(rows)

In [11]:
#valuta un modello di raccomandazione, su val o test
def evaluate_model(samples_df,model_name,score_fn,split_name,k_values=K_VALUES):
    #sampledf df con pos e neg campionati
    ranks=[] #contiene solo rank dei prodotti positivi
    rank_rows=[] #contiene info per ogni esempio
    iterator=samples_df.itertuples(index=False) #trasforma df in iteratore riga per riga
    for row in tqdm(iterator,total=len(samples_df),desc=f"Evaluate {model_name} {split_name}"): #scorre righe in samplesdf
        #restituisce rank la pos nel ranking, e positive_score il punteggio dato
        rank,positive_score=rank_positive(row.user_idx,row.positive_item_idx,row.negative_item_idxs,score_fn)
        ranks.append(rank)
        rank_rows.append({
            "model":model_name,
            "split":split_name,
            "user_idx":int(row.user_idx),
            "positive_item_idx":int(row.positive_item_idx),
            "positive_rank":int(rank),
            "positive_score":float(positive_score),
            "num_negatives":int(len(row.negative_item_idxs)),
        })
    metrics_df=calculate_topk_metrics(ranks,k_values) #calcola le metriche topk partendo dai rank
    metrics_df.insert(0,"split",split_name)
    metrics_df.insert(0,"model",model_name)
    ranks_df=pd.DataFrame(rank_rows)
    return metrics_df,ranks_df

## 5 baseline most popular

In [12]:
#costruiamo e valutiamo la baseline

popularity_series=train_df["item_idx"].value_counts() #conta quante volte un prodotto compare nel training set
popularity_scores={int(item_idx):float(count) for item_idx,count in popularity_series.items()} #trasforma la series in diz python

def score_popularity(user_idx,item_idxs): #funzione di scoring, input utente da valutare, e lista di prodotti candidati
    return np.array([popularity_scores.get(int(item_idx),0.0) for item_idx in item_idxs],dtype=np.float64)
    #output un array con un punteggio per ogni prodotto candidato

model_score_functions={} #salva funzioni di scoring dei modelli
validation_metric_frames=[] #salva metriche aggregate dei modelli su val
validation_rank_frames=[] #salva i rank dettagliati dei modelli su val

if RUN_POPULARITY: 
    model_score_functions["MostPopular"]=score_popularity
    #valuta mostpopular sul validation set
    popularity_validation_metrics,popularity_validation_ranks=evaluate_model(validation_samples_df,"MostPopular",score_popularity,"validation")
    
    validation_metric_frames.append(popularity_validation_metrics) #aggiunge i risultati alle liste generali
    validation_rank_frames.append(popularity_validation_ranks)
    display(popularity_validation_metrics)

Evaluate MostPopular validation:   0%|          | 0/131376 [00:00<?, ?it/s]

Evaluate MostPopular validation:   3%|▎         | 3896/131376 [00:00<00:03, 38954.28it/s]

Evaluate MostPopular validation:   6%|▌         | 7792/131376 [00:00<00:03, 38640.23it/s]

Evaluate MostPopular validation:   9%|▉         | 11657/131376 [00:00<00:03, 36912.71it/s]

Evaluate MostPopular validation:  12%|█▏        | 15357/131376 [00:00<00:03, 36761.87it/s]

Evaluate MostPopular validation:  14%|█▍        | 19038/131376 [00:00<00:03, 36132.45it/s]

... (31 outputs left out)

Evaluate MostPopular validation:  96%|█████████▌| 126370/131376 [00:03<00:00, 31819.12it/s]

Evaluate MostPopular validation:  99%|█████████▊| 129571/131376 [00:04<00:00, 30884.68it/s]

Evaluate MostPopular validation: 100%|██████████| 131376/131376 [00:04<00:00, 31872.36it/s]

,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,5,0.341973,0.341973,0.244906,0.212951,131376
1,MostPopular,validation,10,0.464293,0.464293,0.284317,0.229132,131376
2,MostPopular,validation,20,0.609853,0.609853,0.321042,0.239172,131376


## 6 itemknn
similarità fra prodotti a partire dagli utenti che li hanno valutati positivamente
per un utente un prodotto riceve punteggio alto se è simile a prodotti già presenti nel suo storico di training

In [13]:
#Prepariamo le strutture usate dalle diverse configurazioni ItemKNN
#La similarità viene calcolata direttamente sulle interazioni positive binarie
#Non applichiamo il centraggio perché tutte le interazioni osservate hanno valore 1

itemknn_artifacts={}
itemknn_neighbor_indices=None
itemknn_neighbor_scores=None
empty_int_array=np.array([],dtype=np.int64)
empty_float_array=np.array([],dtype=np.float32)

if RUN_ITEMKNN and n_items>1:
    itemknn_neighbors_list=sorted(set(int(k) for k in ITEMKNN_NEIGHBORS_LIST if int(k)>0))

    if itemknn_neighbors_list:
        item_user_matrix=train_matrix.T.tocsr().astype(np.float32)
        max_itemknn_neighbors=int(min(max(itemknn_neighbors_list)+1,n_items))

        itemknn_model=NearestNeighbors(n_neighbors=max_itemknn_neighbors,metric="cosine",algorithm="brute",n_jobs=ITEMKNN_N_JOBS)
        itemknn_model.fit(item_user_matrix)

        distances,indices=itemknn_model.kneighbors(item_user_matrix,return_distance=True)
        similarities=(1.0-distances).astype(np.float32)

        itemknn_neighbor_indices=indices.astype(np.int64)
        itemknn_neighbor_scores=similarities

        for k in itemknn_neighbors_list:
            model_name=f"ItemKNN_{int(k)}"
            target_neighbors=int(min(int(k),n_items-1))
            current_lookup={}

            for item_idx in range(n_items):
                current_indices=itemknn_neighbor_indices[item_idx]
                current_scores=itemknn_neighbor_scores[item_idx]
                keep_mask=(current_indices!=item_idx)&(current_scores>0)
                filtered_indices=current_indices[keep_mask][:target_neighbors]
                filtered_scores=current_scores[keep_mask][:target_neighbors]
                current_lookup[int(item_idx)]=(filtered_indices,filtered_scores)

            itemknn_artifacts[model_name]={"k":target_neighbors,"neighbor_lookup":current_lookup}

        print("ItemKNN costruito per K:",itemknn_neighbors_list)
        print("Matrice vicini massima:",itemknn_neighbor_indices.shape)

    else:
        print("ItemKNN saltato: lista vicini vuota")

else:
    print("ItemKNN saltato")

ItemKNN costruito per K: [5, 10, 50, 100]
Matrice vicini massima: (12306, 101)


In [14]:
def make_itemknn_score_fn(current_neighbor_lookup): #funzione che riceve una configurazione di vicini
    def score_itemknn(user_idx,item_idxs):  #calcola punteggi itemknn
        #input utente da valutare e prodotti candidati
        history=train_seen_by_user.get(int(user_idx),set())  #recupera prodotti visti dall'utente nel training
        scores=np.zeros(len(item_idxs),dtype=np.float64)
        if not history:
            return scores
        for pos,item_idx in enumerate(item_idxs): #scorre prodotti candidati
            neigh_items,neigh_scores=current_neighbor_lookup.get(int(item_idx),(empty_int_array,empty_float_array))
            #recupera vicini del prodotto candidato
            total_score=0.0
            for neigh_item,neigh_score in zip(neigh_items,neigh_scores): #scorre insieme prodotti vicini e loro punteggi di similarità
                if int(neigh_item) in history: #se un prodotto vicino è stato visto, aggiunge similarità al punteggio
                    total_score+=float(neigh_score)
            scores[pos]=total_score
        return scores
    return score_itemknn

if RUN_ITEMKNN and itemknn_artifacts:
    for model_name,artifact in itemknn_artifacts.items():
        score_fn=make_itemknn_score_fn(artifact["neighbor_lookup"]) #funzione di scoring
        artifact["score_fn"]=score_fn
        model_score_functions[model_name]=score_fn 
        itemknn_validation_metrics,itemknn_validation_ranks=evaluate_model(validation_samples_df,model_name,score_fn,"validation")
        #valuta questa versione sul validation set
        validation_metric_frames.append(itemknn_validation_metrics)
        validation_rank_frames.append(itemknn_validation_ranks)
        print(model_name,"neighbors",artifact["k"])
        display(itemknn_validation_metrics)
else:
    print("ItemKNN validation saltata")

Evaluate ItemKNN_5 validation:   0%|          | 0/131376 [00:00<?, ?it/s]

Evaluate ItemKNN_5 validation:   1%|          | 733/131376 [00:00<00:17, 7323.98it/s]

Evaluate ItemKNN_5 validation:   1%|          | 1466/131376 [00:00<00:19, 6601.56it/s]

Evaluate ItemKNN_5 validation:   2%|▏         | 2177/131376 [00:00<00:18, 6817.11it/s]

Evaluate ItemKNN_5 validation:   2%|▏         | 2863/131376 [00:00<00:19, 6743.35it/s]

Evaluate ItemKNN_5 validation:   3%|▎         | 3540/131376 [00:00<00:18, 6738.47it/s]

... (1,509 outputs left out)

Evaluate ItemKNN_100 validation: 100%|█████████▉| 131352/131376 [01:02<00:00, 2133.45it/s]

Evaluate ItemKNN_100 validation: 100%|██████████| 131376/131376 [01:02<00:00, 2086.21it/s]

ItemKNN_100 neighbors 100


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_100,validation,5,0.110210,0.110210,0.082251,0.073047,131376
1,ItemKNN_100,validation,10,0.152258,0.152258,0.095706,0.078517,131376
2,ItemKNN_100,validation,20,0.240828,0.240828,0.117754,0.084389,131376


## 7 matrix factorization con truncated svd
Truncated SVD apprende rappresentazioni latenti di utenti e prodotti a partire dalla matrice di training
Il punteggio utente-prodotto è il prodotto scalare tra embedding utente ed embedding prodotto

In [15]:
svd_artifacts={} #dizionario per salvare modelli SVD addestrati con diversi numeri di componenti
#funzione che da scoring SVD
def make_svd_score_fn(user_embeddings,item_embeddings): #input vettori latenti degli utenti, vettori latenti dei prodotti
    def score_svd(user_idx,item_idxs): #funzione interna che assegna punteggio ai prodotti latenti per un certo utente
        user_vector=user_embeddings[int(user_idx)] #prende vettore latente dell'utente
        item_matrix=item_embeddings[np.asarray(item_idxs,dtype=np.int64)] 
        #prende vettori latenti dei prodotti candidati, converte lista dei prodotti in array di interi
        #per poter indicizzare item_embeddings
        return item_matrix@user_vector #calcola prodotto scalare tra ogni prodotto candidato e l'utente
    return score_svd #restituisce la funzione

def train_svd_model(n_components): #funzione che addestra modello svd
    max_components=max(1,min(n_users,n_items)-1) #max n di componenti
    actual_components=int(min(int(n_components),max_components)) #quante componenti usare davvero
    svd=TruncatedSVD(n_components=actual_components,n_iter=SVD_N_ITER,random_state=RANDOM_STATE) #crea il modello TruncatedSVD
    #ncomponent = n dim latenti, niter = numero iterazioni
    user_embeddings=svd.fit_transform(train_matrix).astype(np.float32) #addestra svd sulla matrice di training e produce gli embedding degli utenti
    #user embeddings ha forma utenti x componenti
    item_embeddings=svd.components_.T.astype(np.float32) #estrae embeddings dei prodotti
    #matrice del tipo prodotti x componenti 
    score_fn=make_svd_score_fn(user_embeddings,item_embeddings) #funzione di scoring per questo specifico modello SVD
    artifact={
        "svd":svd,
        "user_embeddings":user_embeddings,
        "item_embeddings":item_embeddings,
        "explained_variance_ratio":float(svd.explained_variance_ratio_.sum()),
        "requested_components":int(n_components),
        "actual_components":int(actual_components),
        "score_fn":score_fn,
    }
    return artifact

In [16]:
if RUN_SVD and n_users>1 and n_items>1:
    for n_factors in SVD_FACTORS: #scorre tutti i valori latenti che vogliamo provare
        model_name=f"SVD_{int(n_factors)}"
        artifact=train_svd_model(n_factors) #addestra e restituisce dizionario con i vari dati
        svd_artifacts[model_name]=artifact
        model_score_functions[model_name]=artifact["score_fn"] #salva funzione di scoring
        svd_validation_metrics,svd_validation_ranks=evaluate_model(validation_samples_df,model_name,artifact["score_fn"],"validation") 
        #valuta modello svd sul validation set
        validation_metric_frames.append(svd_validation_metrics) 
        validation_rank_frames.append(svd_validation_ranks)
        print(model_name,"explained_variance_ratio",artifact["explained_variance_ratio"])
        display(svd_validation_metrics)
else:
    print("SVD saltato")

Evaluate SVD_2 validation:   0%|          | 0/131376 [00:00<?, ?it/s]

Evaluate SVD_2 validation:   3%|▎         | 3968/131376 [00:00<00:03, 39675.50it/s]

Evaluate SVD_2 validation:   6%|▌         | 8139/131376 [00:00<00:03, 40871.14it/s]

Evaluate SVD_2 validation:   9%|▉         | 12227/131376 [00:00<00:03, 39503.45it/s]

Evaluate SVD_2 validation:  12%|█▏        | 16183/131376 [00:00<00:02, 38832.50it/s]

Evaluate SVD_2 validation:  15%|█▌        | 20070/131376 [00:00<00:02, 38098.54it/s]

... (312 outputs left out)

Evaluate SVD_32 validation:  99%|█████████▊| 129434/131376 [00:03<00:00, 40195.12it/s]

Evaluate SVD_32 validation: 100%|██████████| 131376/131376 [00:03<00:00, 39662.52it/s]

SVD_32 explained_variance_ratio 0.06471524387598038


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_32,validation,5,0.289771,0.289771,0.206207,0.178681,131376
1,SVD_32,validation,10,0.395810,0.395810,0.240402,0.192737,131376
2,SVD_32,validation,20,0.525903,0.525903,0.273147,0.201648,131376


## 8 scelta modello su validation

In [17]:
if not validation_metric_frames:
    raise RuntimeError("Nessun modello valutato su validation")

validation_metrics_df=pd.concat(validation_metric_frames,ignore_index=True) #unisce
validation_ranks_df=pd.concat(validation_rank_frames,ignore_index=True) if validation_rank_frames else pd.DataFrame()

target_validation_df=validation_metrics_df[validation_metrics_df["K"]==PRIMARY_K].copy() #tiene solo il k scelto come valore principale
target_validation_df=target_validation_df.sort_values(PRIMARY_METRIC,ascending=False).reset_index(drop=True) #ordina dal migliore al peggiore

best_model_name=str(target_validation_df.loc[0,"model"]) 
best_validation_score=float(target_validation_df.loc[0,PRIMARY_METRIC])

print("Best model:",best_model_name)
print("Best validation score:",best_validation_score)
display(validation_metrics_df)
display(target_validation_df)

Best model: MostPopular
Best validation score: 0.2843165188407551


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,5,0.341973,0.341973,0.244906,0.212951,131376
1,MostPopular,validation,10,0.464293,0.464293,0.284317,0.229132,131376
2,MostPopular,validation,20,0.609853,0.609853,0.321042,0.239172,131376
3,ItemKNN_5,validation,5,0.066877,0.066877,0.048103,0.041985,131376
4,ItemKNN_5,validation,10,0.106785,0.106785,0.060816,0.047120,131376
5,ItemKNN_5,validation,20,0.201513,0.201513,0.084356,0.053366,131376
6,ItemKNN_10,validation,5,0.072456,0.072456,0.053376,0.047152,131376
7,ItemKNN_10,validation,10,0.112068,0.112068,0.066000,0.052254,131376
8,ItemKNN_10,validation,20,0.206126,0.206126,0.089378,0.058460,131376
9,ItemKNN_50,validation,5,0.093662,0.093662,0.070877,0.063385,131376


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,10,0.464293,0.464293,0.284317,0.229132,131376
1,SVD_4,validation,10,0.444381,0.444381,0.275201,0.223258,131376
2,SVD_6,validation,10,0.443468,0.443468,0.273863,0.221809,131376
3,SVD_8,validation,10,0.438398,0.438398,0.270965,0.219558,131376
4,SVD_10,validation,10,0.438703,0.438703,0.270385,0.218657,131376
5,SVD_2,validation,10,0.434189,0.434189,0.269721,0.219165,131376
6,SVD_12,validation,10,0.434828,0.434828,0.267599,0.216214,131376
7,SVD_16,validation,10,0.426037,0.426037,0.262517,0.212279,131376
8,SVD_24,validation,10,0.408065,0.408065,0.248977,0.200116,131376
9,SVD_32,validation,10,0.395810,0.395810,0.240402,0.192737,131376


## 9 valutazione su test

In [18]:
test_metric_frames=[]
test_rank_frames=[]

for model_name,score_fn in model_score_functions.items():
    test_metrics,test_ranks=evaluate_model(test_samples_df,model_name,score_fn,"test")
    test_metric_frames.append(test_metrics)
    test_rank_frames.append(test_ranks)
    display(test_metrics)

test_metrics_df=pd.concat(test_metric_frames,ignore_index=True) if test_metric_frames else pd.DataFrame()
test_ranks_df=pd.concat(test_rank_frames,ignore_index=True) if test_rank_frames else pd.DataFrame()

Evaluate MostPopular test:   0%|          | 0/131352 [00:00<?, ?it/s]

Evaluate MostPopular test:   3%|▎         | 3860/131352 [00:00<00:03, 38590.38it/s]

Evaluate MostPopular test:   6%|▌         | 7720/131352 [00:00<00:03, 37533.54it/s]

Evaluate MostPopular test:   9%|▊         | 11476/131352 [00:00<00:03, 37456.34it/s]

Evaluate MostPopular test:  12%|█▏        | 15261/131352 [00:00<00:03, 37608.71it/s]

Evaluate MostPopular test:  15%|█▍        | 19112/131352 [00:00<00:02, 37928.09it/s]

... (2,197 outputs left out)

Evaluate SVD_32 test:  95%|█████████▍| 124722/131352 [00:03<00:00, 35574.19it/s]

Evaluate SVD_32 test:  98%|█████████▊| 128392/131352 [00:03<00:00, 35884.36it/s]

Evaluate SVD_32 test: 100%|██████████| 131352/131352 [00:03<00:00, 33625.11it/s]

,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_32,test,5,0.294453,0.294453,0.209614,0.181673,131352
1,SVD_32,test,10,0.397124,0.397124,0.242729,0.195289,131352
2,SVD_32,test,20,0.524590,0.524590,0.274847,0.204047,131352


In [19]:
all_metrics_df=pd.concat([validation_metrics_df,test_metrics_df],ignore_index=True) #unisce in una unica tabella metriche di val e test

baseline_df=( #tab separata per risultati della baseline most popular
    all_metrics_df[all_metrics_df["model"]=="MostPopular"]
    [["split","K","hitrate","recall","ndcg","mrr"]]
    .rename(columns={
        "hitrate":"hitrate_baseline",
        "recall":"recall_baseline",
        "ndcg":"ndcg_baseline",
        "mrr":"mrr_baseline",
    })
)

comparison_df=all_metrics_df.merge(baseline_df,on=["split","K"],how="left") #unisce tab dei modelli con la tab della baseline

for metric in ["hitrate","recall","ndcg","mrr"]:
    comparison_df[f"{metric}_delta_vs_popularity"]=comparison_df[metric]-comparison_df[f"{metric}_baseline"] #per ogni metrica, calcola quanto è sopra o sotto alla baseline

display(comparison_df.sort_values(["split","K","ndcg"],ascending=[True,True,False]))

,model,split,K,hitrate,recall,ndcg,mrr,n_examples,hitrate_baseline,recall_baseline,ndcg_baseline,mrr_baseline,hitrate_delta_vs_popularity,recall_delta_vs_popularity,ndcg_delta_vs_popularity,mrr_delta_vs_popularity
42,MostPopular,test,5,0.361639,0.361639,0.261726,0.228795,131352,0.361639,0.361639,0.261726,0.228795,0.000000,0.000000,0.000000,0.000000
60,SVD_4,test,5,0.355267,0.355267,0.259025,0.227282,131352,0.361639,0.361639,0.261726,0.228795,-0.006372,-0.006372,-0.002701,-0.001513
63,SVD_6,test,5,0.354132,0.354132,0.256577,0.224405,131352,0.361639,0.361639,0.261726,0.228795,-0.007507,-0.007507,-0.005148,-0.004389
66,SVD_8,test,5,0.349816,0.349816,0.253429,0.221642,131352,0.361639,0.361639,0.261726,0.228795,-0.011823,-0.011823,-0.008297,-0.007152
57,SVD_2,test,5,0.345126,0.345126,0.251190,0.220232,131352,0.361639,0.361639,0.261726,0.228795,-0.016513,-0.016513,-0.010536,-0.008562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41,SVD_32,validation,20,0.525903,0.525903,0.273147,0.201648,131376,0.609853,0.609853,0.321042,0.239172,-0.083950,-0.083950,-0.047895,-0.037524
14,ItemKNN_100,validation,20,0.240828,0.240828,0.117754,0.084389,131376,0.609853,0.609853,0.321042,0.239172,-0.369025,-0.369025,-0.203288,-0.154783
11,ItemKNN_50,validation,20,0.224211,0.224211,0.106214,0.074588,131376,0.609853,0.609853,0.321042,0.239172,-0.385641,-0.385641,-0.214828,-0.164584
8,ItemKNN_10,validation,20,0.206126,0.206126,0.089378,0.058460,131376,0.609853,0.609853,0.321042,0.239172,-0.403727,-0.403727,-0.231664,-0.180712


## 10 salvataggio risultati

In [20]:
METRICS_PATH=NOTEBOOK04_DIR/"notebook04_metrics.csv"
COMPARISON_PATH=NOTEBOOK04_DIR/"notebook04_comparison.csv"
VALIDATION_RANKS_PATH=NOTEBOOK04_DIR/"notebook04_validation_ranks.parquet"
TEST_RANKS_PATH=NOTEBOOK04_DIR/"notebook04_test_ranks.parquet"
SUMMARY_PATH=NOTEBOOK04_DIR/"notebook04_summary.json"
POPULARITY_SCORES_PATH=NOTEBOOK04_DIR/"popularity_scores.parquet"
ITEMKNN_INDICES_PATH=NOTEBOOK04_DIR/"itemknn_neighbor_indices.npy"
ITEMKNN_SCORES_PATH=NOTEBOOK04_DIR/"itemknn_neighbor_scores.npy"
SVD_USER_EMB_PATH=NOTEBOOK04_DIR/"best_svd_user_embeddings.npy"
SVD_ITEM_EMB_PATH=NOTEBOOK04_DIR/"best_svd_item_embeddings.npy"
SVD_INFO_PATH=NOTEBOOK04_DIR/"best_svd_info.json"

popularity_scores_df=pd.DataFrame({
    "item_idx":list(popularity_scores.keys()),
    "popularity_score":list(popularity_scores.values()),
})


In [21]:
all_metrics_df.to_csv(METRICS_PATH,index=False) #salva tutte le metriche
comparison_df.to_csv(COMPARISON_PATH,index=False) #salva tab confronto con baseline
popularity_scores_df.to_parquet(POPULARITY_SCORES_PATH,index=False) #salva punteggi popolarità

if SAVE_RANKS:
    validation_ranks_df.to_parquet(VALIDATION_RANKS_PATH,index=False)
    test_ranks_df.to_parquet(TEST_RANKS_PATH,index=False)

if SAVE_MODEL_ARTIFACTS and itemknn_neighbor_indices is not None:
    np.save(ITEMKNN_INDICES_PATH,itemknn_neighbor_indices)
    np.save(ITEMKNN_SCORES_PATH,itemknn_neighbor_scores)

best_svd_name=None
if SAVE_MODEL_ARTIFACTS and svd_artifacts:
    svd_candidates=target_validation_df[target_validation_df["model"].isin(svd_artifacts.keys())]
    if not svd_candidates.empty:
        best_svd_name=str(svd_candidates.iloc[0]["model"])
        best_svd_artifact=svd_artifacts[best_svd_name]
        np.save(SVD_USER_EMB_PATH,best_svd_artifact["user_embeddings"])
        np.save(SVD_ITEM_EMB_PATH,best_svd_artifact["item_embeddings"])
        SVD_INFO_PATH.write_text(json.dumps({
            "model":best_svd_name,
            "requested_components":best_svd_artifact["requested_components"],
            "actual_components":best_svd_artifact["actual_components"],
            "explained_variance_ratio":best_svd_artifact["explained_variance_ratio"],
        },indent=2),encoding="utf-8")

summary={
    "category":CATEGORY,
    "primary_metric":PRIMARY_METRIC,
    "primary_k":PRIMARY_K,
    "best_model_validation":best_model_name,
    "best_validation_score":best_validation_score,
    "models_evaluated":list(model_score_functions.keys()),
    "train_rows":int(len(train_df)),
    "validation_samples":int(len(validation_samples_df)),
    "test_samples":int(len(test_samples_df)),
    "n_users":int(n_users),
    "n_items":int(n_items),
    "metrics_path":str(METRICS_PATH),
    "comparison_path":str(COMPARISON_PATH),
    "popularity_scores_path":str(POPULARITY_SCORES_PATH),
    "best_svd_name":best_svd_name,
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2),encoding="utf-8")

display(pd.DataFrame.from_dict(summary,orient="index",columns=["value"]))

,value
category,Electronics
primary_metric,ndcg
primary_k,10
best_model_validation,MostPopular
best_validation_score,0.284317
models_evaluated,"[MostPopular, ItemKNN_5, ItemKNN_10, ItemKNN_50, ItemKNN_100, SVD_2, SVD_4, SVD_6, SVD_8, SVD_10, SVD_12, SVD_16, SV..."
train_rows,287752
validation_samples,131376
test_samples,131352
n_users,131398


## 11 controlli qualitativi

In [22]:
def top_items_for_user(user_idx,score_fn,top_n=DEMO_TOP_N):
    #funzione che restituisce i migliori top_n prodotti per un certo utente
    seen_items=all_seen_by_user.get(int(user_idx),set())
    candidate_items=np.array([item for item in train_item_candidates if int(item) not in seen_items],dtype=np.int64)
    #prodotti candidati
    if len(candidate_items)==0:
        return pd.DataFrame()
    scores=score_fn(int(user_idx),candidate_items) #calcola punteggi di ogni prodotto candidato
    scores=np.asarray(scores,dtype=np.float64)
    scores=np.nan_to_num(scores,nan=-np.inf,posinf=np.inf,neginf=-np.inf)
    order=np.lexsort((candidate_items,-scores))
    top_positions=order[:int(top_n)]
    return pd.DataFrame({
        "user_idx":int(user_idx),
        "rank":np.arange(1,len(top_positions)+1,dtype=np.int64),
        "item_idx":candidate_items[top_positions].astype(int),
        "score":scores[top_positions].astype(float),
    })

def attach_item_metadata(recommendations_df): #funzione che aggiunge info sui prodotti alle raccomandazioni
    if recommendations_df.empty:
        return recommendations_df
    items_for_join=items_df.copy()
    if "item_idx" not in items_for_join.columns:
        items_for_join=items_for_join.merge(item_mapping_df,on="parent_asin",how="left")
    return recommendations_df.merge(items_for_join,on="item_idx",how="left")

In [23]:
demo_user_idxs=(
    test_samples_df["user_idx"]
    .drop_duplicates()
    .head(DEMO_USERS_N) #tiene solo i primi utenti, per demo test
    .astype(int)
    .tolist()
)

best_score_fn=model_score_functions[best_model_name] #recupera funzione migliore, per ora baseline
demo_frames=[]

for user_idx in demo_user_idxs:
    demo_frames.append(top_items_for_user(user_idx,best_score_fn,DEMO_TOP_N)) #per ogni utente genera le raccomandazioni topN

demo_recommendations_df=pd.concat(demo_frames,ignore_index=True) if demo_frames else pd.DataFrame()
demo_recommendations_df=attach_item_metadata(demo_recommendations_df) 

DEMO_RECOMMENDATIONS_PATH=NOTEBOOK04_DIR/"demo_recommendations.csv"
demo_recommendations_df.to_csv(DEMO_RECOMMENDATIONS_PATH,index=False)

preferred_cols=["user_idx","rank","item_idx","parent_asin","title","main_category","average_rating","rating_number","score"]
display_cols=[col for col in preferred_cols if col in demo_recommendations_df.columns]

display(demo_recommendations_df[display_cols].head(30) if display_cols else demo_recommendations_df.head(30))
print("Demo recommendations path:",DEMO_RECOMMENDATIONS_PATH)

,user_idx,rank,item_idx,parent_asin,title,score
0,0,1,5996,B0052SCU8U,"Amazon Basics High-Speed HDMI Cable 2-Pack - 6.5 Feet (2 Meters) Supports Ethernet, 3D, 4K and Audio Return",1118.0
1,0,2,2076,B0019EHU8G,"Mediabridge™ HDMI Cable (6 Feet) Supports 4K@60Hz, High Speed, Hand-Tested, HDMI 2.0 Ready - UHD, 18Gbps, Audio Retu...",1103.0
2,0,3,4800,B0043T7FXE,"Logitech M570 Wireless Trackball Mouse – Ergonomic Design with Sculpted Right-Hand Shape, Compatible with Apple Mac ...",898.0
3,0,4,10328,B00BWF5U0M,SABRENT 4 Port USB 2.0 Data Hub with Individual LED lit Power Switches [Charging NOT Supported] for Mac & PC (HB-UMLS),898.0
4,0,5,12171,B00IVPU786,Samsung 32GB up to 48MB/s EVO Class 10 Micro SDHC Card with Adapter (MB-MP32DA/AM),780.0
5,0,6,4310,B003NR57BY,"Logitech M510 Wireless Computer Mouse – Comfortable Shape with USB Unifying Receiver, with Back/Forward Buttons and ...",757.0
6,0,7,12172,B00IVPU7AO,Samsung EVO 64GB 48/MB/s Micro SDXC Memory Card with Adapter up to (MB-MP64DA/AM),727.0
7,0,8,7056,B006JH8T3S,"Logitech HD Pro Webcam C920, Widescreen Video Calling and Recording, 1080p Camera, Desktop or Laptop Webcam (Discont...",713.0
8,0,9,4278,B003MTTJOY,"Edimax EW-7811Un 150Mbps 11n Wi-Fi USB Adapter, Nano Size Lets You Plug it and Forget it, Ideal for Raspberry Pi / P...",676.0
9,0,10,3497,B002VPE1WK,Amazon Basics Backpack for SLR Cameras and Accessories-Black,632.0


Demo recommendations path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\notebook04_models\demo_recommendations.csv
